In [1]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import pandas as pd
import os
import numpy as np

In [2]:
#Obtencion de servidor Sql

load_dotenv()  # ← carga el .env

DB_SERVER = os.getenv("DB_SERVER")
DB_NAME = os.getenv("DB_NAME")

In [3]:
# Conectar la info del API al SQL y guardar los datos en la base de datos para su posterior análisis.
engine = create_engine(
    f"mssql+pyodbc://{DB_SERVER}/{DB_NAME}"
    f"?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

In [4]:
partidos = pd.read_sql('SELECT * FROM fact_partido',con=engine)

c:\ProgramData\anaconda3\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


In [9]:
print(np.round((partidos[['temporada',"HS", "AS", "HST", "AST", "HC", "AC", "HF", "AF", "HY", "AY", "HR", "AR"]].isnull().sum()/len(partidos)*100),2))

temporada     0.00
HS           19.45
AS           19.45
HST          19.45
AST          19.45
HC           19.45
AC           19.45
HF           19.45
AF           19.45
HY           19.45
AY           19.45
HR           19.45
AR           19.45
dtype: float64


In [10]:
datos_nulos = partidos[['temporada',"HS", "AS", "HST", "AST", "HC", "AC", "HF", "AF", "HY", "AY", "HR", "AR"]]
for temp in datos_nulos.temporada.unique():
    subset = datos_nulos[datos_nulos['temporada']== temp]
    print('Para la campaña',temp)
    print(np.round((subset.isnull().sum()/len(subset)*100),2))

Para la campaña 0001
temporada      0.0
HS           100.0
AS           100.0
HST          100.0
AST          100.0
HC           100.0
AC           100.0
HF           100.0
AF           100.0
HY           100.0
AY           100.0
HR           100.0
AR           100.0
dtype: float64
Para la campaña 0102
temporada      0.0
HS           100.0
AS           100.0
HST          100.0
AST          100.0
HC           100.0
AC           100.0
HF           100.0
AF           100.0
HY           100.0
AY           100.0
HR           100.0
AR           100.0
dtype: float64
Para la campaña 0203
temporada      0.0
HS           100.0
AS           100.0
HST          100.0
AST          100.0
HC           100.0
AC           100.0
HF           100.0
AF           100.0
HY           100.0
AY           100.0
HR           100.0
AR           100.0
dtype: float64
Para la campaña 0304
temporada      0.0
HS           100.0
AS           100.0
HST          100.0
AST          100.0
HC           100.0
AC           100

In [13]:
filtro_campaña = ['0001', '0102', '0203', '0304', '0405']

partidos = partidos[~partidos['temporada'].isin(filtro_campaña)]

In [15]:
partidos[['temporada','B365A','B365D','B365H']].isnull().sum()

temporada    0
B365A        0
B365D        0
B365H        0
dtype: int64

In [34]:
df_fe = partidos.copy()

In [36]:
#Datos de los equipos Locales
data_local = df_fe[['temporada','Date','id_partido','HomeTeam','FTHG','FTAG','HS','HST','HC','HF','HY','HR','HTHG','HTAG','FTR']]\
            .rename(columns={'temporada':'temporada','Date':'Date','id_partido':'id_partido','HomeTeam':'Equipo','FTHG':'Goles','FTAG':'goles_rival','HS':'tiros',
                             'HST':'tiros_al_arco','HC':'corners','HF':'faltas','HY':'amarillas',
                             'HR':'rojas','HTHG':'goles_ht','HTAG':'goles_ht_rival','FTR':'resultado'})
data_local['Condicion'] = 'Local'

#Datos de los equipos Visitantes
data_visitante = df_fe[['temporada','Date','id_partido','AwayTeam','FTAG','FTHG','AS','AST','AC','AF','AY','AR','HTAG','HTHG','FTR']]\
            .rename(columns={'temporada':'temporada','Date':'Date','id_partido':'id_partido','AwayTeam':'Equipo','FTAG':'Goles','FTHG':'goles_rival','AS':'tiros',
                             'AST':'tiros_al_arco','AC':'corners','AF':'faltas','AY':'amarillas',
                             'AR':'rojas','HTAG':'goles_ht','HTHG':'goles_ht_rival','FTR':'resultado'})
data_visitante['Condicion'] = 'Visitantes'

In [37]:
# Logic: If Result was 'V' (local Win), for the Visitor it's 'D' (Loss)
res_local     = {'H': 3, 'D': 1, 'A': 0}
res_visitante = {'A': 3, 'D': 1, 'H': 0}
data_local['resultado'] = data_local['resultado'].map(res_local)
data_visitante['resultado'] = data_visitante['resultado'].map(res_visitante)

In [38]:
df_partidos = pd.concat([data_local, data_visitante], ignore_index=True)

In [41]:
df_partidos[df_partidos['id_partido'] == 1901]

,temporada,Date,id_partido,Equipo,Goles,goles_rival,tiros,tiros_al_arco,corners,faltas,amarillas,rojas,goles_ht,goles_ht_rival,resultado,Condicion
0,0506,2005-08-27,1901,Alaves,0,0,5.0,0.0,3.0,17.0,0.0,0.0,0,0,1,Local
7870,0506,2005-08-27,1901,Barcelona,0,0,17.0,10.0,7.0,19.0,1.0,0.0,0,0,1,Visitantes


In [43]:
len(df_fe),len(df_partidos)

(7870, 15740)

In [44]:
df_partidos.columns

Index(['temporada', 'Date', 'id_partido', 'Equipo', 'Goles', 'goles_rival',
       'tiros', 'tiros_al_arco', 'corners', 'faltas', 'amarillas', 'rojas',
       'goles_ht', 'goles_ht_rival', 'resultado', 'Condicion'],
      dtype='object')

In [47]:
df_partidos = df_partidos.sort_values(['Equipo', 'Date'])

In [53]:
col = ['Goles', 'goles_rival',
       'tiros', 'tiros_al_arco', 'corners', 'faltas', 'amarillas', 'rojas',
       'goles_ht', 'goles_ht_rival', 'resultado']
for c in col:
    df_partidos[f'{c}_avg5'] = (
    df_partidos.groupby('Equipo')[c]
    .transform(lambda x: x.rolling(5).mean().shift(1))
)

In [ ]:
df_partidos.tail(10)D

,temporada,Date,id_partido,Equipo,Goles,goles_rival,tiros,tiros_al_arco,corners,faltas,...,goles_rival_avg5,tiros_avg5,tiros_al_arco_avg5,corners_avg5,faltas_avg5,amarillas_avg5,rojas_avg5,goles_ht_avg5,goles_ht_rival_avg5,resultado_avg5
2943,1213,2013-03-30,4844,Zaragoza,1,1,12.0,5.0,3.0,17.0,...,1.8,10.6,3.4,4.2,16.4,3.8,0.6,0.4,0.8,0.4
10822,1213,2013-04-06,4853,Zaragoza,2,3,7.0,3.0,4.0,18.0,...,1.8,11.0,3.8,4.0,17.2,3.0,0.6,0.6,0.8,0.6
2968,1213,2013-04-14,4869,Zaragoza,0,3,12.0,2.0,2.0,16.0,...,2.0,9.6,3.2,3.8,17.2,2.6,0.6,0.6,1.0,0.4
10849,1213,2013-04-22,4880,Zaragoza,1,2,10.0,3.0,5.0,13.0,...,2.2,9.6,3.0,2.8,16.4,2.8,0.4,0.6,1.2,0.4
2984,1213,2013-04-27,4885,Zaragoza,3,2,13.0,6.0,7.0,16.0,...,2.6,10.0,3.0,3.2,16.0,3.0,0.4,0.8,1.4,0.2
2998,1213,2013-05-05,4899,Zaragoza,3,0,16.0,6.0,5.0,14.0,...,2.2,10.8,3.8,4.2,16.0,3.4,0.2,1.0,1.4,0.8
10872,1213,2013-05-10,4903,Zaragoza,0,0,16.0,3.0,9.0,18.0,...,2.0,11.6,4.0,4.6,15.4,3.6,0.2,1.0,1.2,1.2
3018,1213,2013-05-19,4919,Zaragoza,1,2,14.0,3.0,3.0,21.0,...,1.4,13.4,4.0,5.6,15.4,3.6,0.0,0.6,0.8,1.4
10892,1213,2013-05-26,4923,Zaragoza,0,4,16.0,3.0,4.0,12.0,...,1.2,13.8,4.2,5.8,16.4,4.0,0.0,0.8,0.4,1.4
3039,1213,2013-06-01,4940,Zaragoza,1,3,15.0,6.0,7.0,11.0,...,1.6,15.0,4.2,5.6,16.2,4.4,0.0,0.6,0.6,1.4


In [57]:
df_partidos_local = df_partidos[df_partidos['Condicion']== 'Local']
df_partidos_visitante = df_partidos[df_partidos['Condicion']== 'Visitantes']

In [59]:
col = ['temporada', 'Date', 'id_partido', 'Equipo',
       'Goles_avg5', 'goles_rival_avg5',
       'tiros_avg5', 'tiros_al_arco_avg5',
       'corners_avg5', 'faltas_avg5',
       'amarillas_avg5', 'rojas_avg5',
       'goles_ht_avg5', 'goles_ht_rival_avg5',
       'resultado_avg5']
df_partidos_local = df_partidos_local[col]
df_partidos_visitante = df_partidos_visitante[col]

df_avg = df_partidos_local.merge(
    df_partidos_visitante,
    on='id_partido',
    suffixes=('_local', '_visitante')
)

In [63]:
df_avg

,temporada_local,Date_local,id_partido,Equipo_local,Goles_avg5_local,goles_rival_avg5_local,tiros_avg5_local,tiros_al_arco_avg5_local,corners_avg5_local,faltas_avg5_local,...,goles_rival_avg5_visitante,tiros_avg5_visitante,tiros_al_arco_avg5_visitante,corners_avg5_visitante,faltas_avg5_visitante,amarillas_avg5_visitante,rojas_avg5_visitante,goles_ht_avg5_visitante,goles_ht_rival_avg5_visitante,resultado_avg5_visitante
0,0506,2005-08-27,1901,Alaves,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0506,2005-09-18,1923,Alaves,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0506,2005-09-25,1946,Alaves,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0506,2005-10-15,1961,Alaves,1.2,2.0,11.2,3.6,4.8,14.8,...,1.2,13.4,4.0,4.4,15.2,2.4,0.2,0.6,0.8,1.6
4,0506,2005-10-27,1986,Alaves,0.6,1.4,10.4,3.4,5.8,15.0,...,1.4,9.4,2.8,3.8,15.6,2.4,0.6,0.2,0.2,0.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7865,1213,2013-04-14,4869,Zaragoza,0.6,2.0,9.6,3.2,3.8,17.2,...,1.0,13.2,5.6,4.6,13.2,2.6,0.2,1.8,0.4,2.0
7866,1213,2013-04-27,4885,Zaragoza,0.8,2.6,10.0,3.0,3.2,16.0,...,2.8,14.6,5.0,6.6,16.8,2.4,0.0,0.8,1.4,0.8
7867,1213,2013-05-05,4899,Zaragoza,1.4,2.2,10.8,3.8,4.2,16.0,...,1.6,13.4,5.0,5.8,12.6,3.6,0.0,0.8,1.2,1.0
7868,1213,2013-05-19,4919,Zaragoza,1.4,1.4,13.4,4.0,5.6,15.4,...,1.6,10.4,2.8,4.8,16.6,3.4,0.2,0.8,0.4,1.2


In [73]:
df_unir = df_fe[['id_partido','B365H', 'B365D',
       'B365A', 'FTR']]

df_avg = df_avg.merge(df_unir,
            on='id_partido',
            how='right')

In [74]:
df_avg.columns

Index(['temporada_local', 'Date_local', 'id_partido', 'Equipo_local',
       'Goles_avg5_local', 'goles_rival_avg5_local', 'tiros_avg5_local',
       'tiros_al_arco_avg5_local', 'corners_avg5_local', 'faltas_avg5_local',
       'amarillas_avg5_local', 'rojas_avg5_local', 'goles_ht_avg5_local',
       'goles_ht_rival_avg5_local', 'resultado_avg5_local',
       'temporada_visitante', 'Date_visitante', 'Equipo_visitante',
       'Goles_avg5_visitante', 'goles_rival_avg5_visitante',
       'tiros_avg5_visitante', 'tiros_al_arco_avg5_visitante',
       'corners_avg5_visitante', 'faltas_avg5_visitante',
       'amarillas_avg5_visitante', 'rojas_avg5_visitante',
       'goles_ht_avg5_visitante', 'goles_ht_rival_avg5_visitante',
       'resultado_avg5_visitante', 'B365H', 'B365D', 'B365A', 'FTR'],
      dtype='object')

In [76]:
col_  = ['temporada_local', 'Date_local']

df_avg = df_avg.drop(columns=col_)\
    .rename(columns={'temporada_visitante':'temporada', 'Date_visitante':'Date'})

In [79]:
df_avg.isnull().sum().sort_values(ascending=False)

goles_ht_rival_avg5_visitante    104
amarillas_avg5_visitante         104
corners_avg5_visitante           104
tiros_al_arco_avg5_visitante     104
tiros_avg5_visitante             104
goles_rival_avg5_visitante       104
Goles_avg5_visitante             104
rojas_avg5_visitante             104
goles_ht_avg5_visitante          104
resultado_avg5_visitante         104
faltas_avg5_visitante            104
goles_ht_avg5_local              101
resultado_avg5_local             101
rojas_avg5_local                 101
amarillas_avg5_local             101
faltas_avg5_local                101
corners_avg5_local               101
tiros_al_arco_avg5_local         101
tiros_avg5_local                 101
goles_rival_avg5_local           101
Goles_avg5_local                 101
goles_ht_rival_avg5_local        101
id_partido                         0
B365H                              0
B365D                              0
B365A                              0
Equipo_visitante                   0
E

In [82]:
df_avg = df_avg.dropna()
df_avg.shape

(7717, 31)

* Se retiraron las campañas del 2000 al 2005 debido que no tenia stats de los partidos, que servirian para poder tener indicadores de los ultimos 5 partidos que realizo el equipo
* Se pivoteo los datos en local y visitante para tener los ultimos 5 partidos y despues consolidarlo. se toma en cuenta un rolling continuo ya que sino los primeros 5 partidos de cada campaña estaria vacio
* Se re-ensamblaja los datos del local y visitante para tener una sola sola de fila de informacion del partido
* Se agrega a la informacion consolidada los datos de apuesta de B365 y le resultado del partido
* se elimino 153 filas que corresponden al primera campaña porque no se tiene datos anteriores a ello esto representa unicamente alrededor del 2% lo cual no genera problemas